# 04C. Multi-Event Satellite Data Collection (Full Feature Set)

Downloads all satellite layers needed to build the final ML dataset:

| Layer | Source | Type | Downloaded Once or Per-Event? |
|---|---|---|---|
| Elevation_m + Slope_deg | NASA SRTM | Topography | **Once** (already done in 03B) |
| Rainfall_mm | CHIRPS Daily | Rainfall | **Per Event** |
| Flood_Target | Sentinel-1 SAR | Flood Mask | **Per Event** |
| NDVI | MODIS MOD13A2 | Vegetation | **Per Event** |
| LandCover | ESA WorldCover 2021 | Land Use | **Once** (static) |

In [1]:
import ee
import geopandas as gpd
import requests
import json
import os
from pathlib import Path

try:
    ee.Initialize()
    print('Earth Engine initialized.')
except Exception as e:
    print('Please run notebook 02 first.')
    raise e

notebook_dir  = Path(os.path.abspath(''))
project_root  = notebook_dir.parent
boundary_path = project_root / 'data' / 'raw' / 'chikwawa_boundary.geojson'
output_dir    = project_root / 'data' / 'raw' / 'events'
static_dir    = project_root / 'data' / 'raw' / 'static'
static_dir.mkdir(parents=True, exist_ok=True)

chikwawa_gdf = gpd.read_file(boundary_path)
geojson = json.loads(chikwawa_gdf.to_json())
aoi = ee.FeatureCollection(geojson).geometry()
print(f'AOI loaded. Static layers -> {static_dir}')


Earth Engine initialized.
AOI loaded. Static layers -> c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\static


In [2]:
# All events with full feature set
EVENTS = [
    {
        'name':       'cyclone_bansi_jan2015',
        'label':      'Cyclone Bansi (Jan 2015) - MAJOR',
        'is_flood':   True,
        'rain_start': '2015-01-01', 'rain_end':   '2015-01-31',
        'sar_before': ('2014-11-01', '2014-12-31'),
        'sar_during': ('2015-01-13', '2015-01-28'),
        'ndvi_date':  ('2015-01-01', '2015-02-15'),
    },
    {
        'name':       'floods_feb2017',
        'label':      'Moderate Floods (Feb 2017) - MINOR',
        'is_flood':   True,
        'rain_start': '2017-01-15', 'rain_end':   '2017-02-28',
        'sar_before': ('2016-11-01', '2016-12-31'),
        'sar_during': ('2017-02-05', '2017-02-25'),
        'ndvi_date':  ('2017-01-15', '2017-03-01'),
    },
    {
        'name':       'cyclone_idai_mar2019',
        'label':      'Cyclone Idai (Mar 2019) - MAJOR',
        'is_flood':   True,
        'rain_start': '2019-03-01', 'rain_end':   '2019-03-31',
        'sar_before': ('2019-01-01', '2019-02-28'),
        'sar_during': ('2019-03-10', '2019-03-25'),
        'ndvi_date':  ('2019-02-15', '2019-04-01'),
    },
    {
        'name':       'cyclone_ana_jan2022',
        'label':      'Cyclone Ana (Jan 2022) - MAJOR',
        'is_flood':   True,
        'rain_start': '2022-01-01', 'rain_end':   '2022-01-31',
        'sar_before': ('2021-11-01', '2021-12-31'),
        'sar_during': ('2022-01-20', '2022-02-10'),
        'ndvi_date':  ('2022-01-01', '2022-02-15'),
    },
    {
        'name':       'cyclone_freddy_mar2023',
        'label':      'Cyclone Freddy (Mar 2023) - MAJOR',
        'is_flood':   True,
        'rain_start': '2023-02-01', 'rain_end':   '2023-03-31',
        'sar_before': ('2022-12-01', '2023-01-31'),
        'sar_during': ('2023-03-05', '2023-03-20'),
        'ndvi_date':  ('2023-02-01', '2023-04-01'),
    },
    {
        'name':       'normal_season_2016',
        'label':      'Normal Season (Jan-Feb 2016)',
        'is_flood':   False,
        'rain_start': '2016-01-01', 'rain_end':   '2016-02-29',
        'sar_before': ('2015-10-01', '2015-12-31'),
        'sar_during': ('2016-01-01', '2016-02-29'),
        'ndvi_date':  ('2016-01-01', '2016-03-01'),
    },
    {
        'name':       'normal_season_2018',
        'label':      'Normal Season (Jan-Feb 2018)',
        'is_flood':   False,
        'rain_start': '2018-01-01', 'rain_end':   '2018-02-28',
        'sar_before': ('2017-10-01', '2017-12-31'),
        'sar_during': ('2018-01-01', '2018-02-28'),
        'ndvi_date':  ('2018-01-01', '2018-03-01'),
    },
    {
        'name':       'normal_season_2020',
        'label':      'Normal Season (Jan-Feb 2020)',
        'is_flood':   False,
        'rain_start': '2020-01-01', 'rain_end':   '2020-02-29',
        'sar_before': ('2019-10-01', '2019-12-31'),
        'sar_during': ('2020-01-01', '2020-02-29'),
        'ndvi_date':  ('2020-01-01', '2020-03-01'),
    },
    {
        'name':       'normal_season_2021',
        'label':      'Normal Season (Jan-Feb 2021)',
        'is_flood':   False,
        'rain_start': '2021-01-01', 'rain_end':   '2021-02-28',
        'sar_before': ('2020-10-01', '2020-12-31'),
        'sar_during': ('2021-01-01', '2021-02-28'),
        'ndvi_date':  ('2021-01-01', '2021-03-01'),
    },
   
]
print(f'Defined {len(EVENTS)} events.')



Defined 9 events.


In [3]:
# Helper: Download any EE image directly to disk
def download_ee_image(image, aoi, scale, out_path):
    url = image.getDownloadURL({
        'region': aoi,
        'scale':  scale,
        'format': 'GEO_TIFF',
        'crs':    'EPSG:4326'
    })
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(out_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    return Path(out_path).stat().st_size / (1024 * 1024)


In [4]:
# STEP A: Download static layers (only need to be downloaded ONCE)

# --- ESA WorldCover 2021 (LandCover) ---
lc_path = static_dir / 'landcover_esa2021.tif'
if not lc_path.exists():
    print('Downloading ESA WorldCover 2021 (LandCover)...')
    worldcover = ee.ImageCollection('ESA/WorldCover/v200') \
                   .first() \
                   .select('Map') \
                   .clip(aoi)
    mb = download_ee_image(worldcover, aoi, scale=90, out_path=lc_path)
    print(f'  LandCover saved! ({mb:.2f} MB)')
else:
    print('LandCover already exists, skipping.')

print('Static layers ready.')


  LandCover saved! (0.25 MB)
Static layers ready.


In [5]:
# STEP B: Per-event downloads (Rainfall + Flood Mask + NDVI)
results = []
total   = len(EVENTS)

for i, event in enumerate(EVENTS, 1):
    name  = event['name']
    label = event['label']
    print(f'\n[{i}/{total}] {label}')

    event_dir = output_dir / name
    event_dir.mkdir(exist_ok=True)
    status = {'event': label, 'flood': event['is_flood']}

    # 1. Rainfall (CHIRPS)
    rain_path = event_dir / 'rainfall.tif'
    if not rain_path.exists():
        print(f'  [1/3] CHIRPS Rainfall...')
        try:
            chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
                       .filterDate(event['rain_start'], event['rain_end']) \
                       .select('precipitation')
            mb = download_ee_image(chirps.sum().clip(aoi), aoi, scale=1000, out_path=rain_path)
            print(f'       OK ({mb:.2f} MB)')
            status['rainfall'] = 'OK'
        except Exception as ex:
            print(f'       ERROR: {ex}')
            status['rainfall'] = 'FAILED'
    else:
        print(f'  [1/3] Rainfall already exists, skipping.')
        status['rainfall'] = 'EXISTS'

    # 2. Flood Mask (Sentinel-1 SAR)
    flood_path = event_dir / 'flood_mask.tif'
    if not flood_path.exists():
        print(f'  [2/3] Sentinel-1 SAR Flood Mask...')
        try:
            col = ee.ImageCollection('COPERNICUS/S1_GRD') \
                    .filterBounds(aoi) \
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
                    .select('VV')
            before    = col.filterDate(event['sar_before'][0], event['sar_before'][1]).median().focal_median(50, 'circle', 'meters').clip(aoi)
            during    = col.filterDate(event['sar_during'][0], event['sar_during'][1]).median().focal_median(50, 'circle', 'meters').clip(aoi)
            threshold = 3 if event['is_flood'] else 999
            mask      = before.subtract(during).gt(threshold)
            mb = download_ee_image(mask, aoi, scale=90, out_path=flood_path)
            print(f'       OK ({mb:.2f} MB)')
            status['flood_mask'] = 'OK'
        except Exception as ex:
            print(f'       ERROR: {ex}')
            status['flood_mask'] = 'FAILED'
    else:
        print(f'  [2/3] Flood mask already exists, skipping.')
        status['flood_mask'] = 'EXISTS'

    # 3. NDVI (MODIS 16-day composite at 500m)
    ndvi_path = event_dir / 'ndvi.tif'
    if not ndvi_path.exists():
        print(f'  [3/3] MODIS NDVI ({event["ndvi_date"][0]} to {event["ndvi_date"][1]})...')
        try:
            modis = ee.ImageCollection('MODIS/061/MOD13A2') \
                      .filterDate(event['ndvi_date'][0], event['ndvi_date'][1]) \
                      .select('NDVI') \
                      .mean() \
                      .multiply(0.0001)  # Scale factor: raw NDVI is x10000
            ndvi  = modis.clip(aoi)
            mb = download_ee_image(ndvi, aoi, scale=500, out_path=ndvi_path)
            print(f'       OK ({mb:.2f} MB)')
            status['ndvi'] = 'OK'
        except Exception as ex:
            print(f'       ERROR: {ex}')
            status['ndvi'] = 'FAILED'
    else:
        print(f'  [3/3] NDVI already exists, skipping.')
        status['ndvi'] = 'EXISTS'

    results.append(status)

# Final Summary
print('\n' + '='*60)
print('DOWNLOAD COMPLETE - SUMMARY')
print('='*60)
for r in results:
    tag  = 'FLOOD ' if r['flood'] else 'NORMAL'
    rain = '✓' if r.get('rainfall')  in ('OK','EXISTS') else '✗'
    mask = '✓' if r.get('flood_mask') in ('OK','EXISTS') else '✗'
    ndvi = '✓' if r.get('ndvi')       in ('OK','EXISTS') else '✗'
    print(f'  [{tag}] {r["event"]}')
    print(f'          Rain:{rain}  FloodMask:{mask}  NDVI:{ndvi}')
print('='*60)
print('Ready for Notebook 05 - Feature Extraction!')




[1/9] Cyclone Bansi (Jan 2015) - MAJOR
  [1/3] CHIRPS Rainfall...
       OK (0.00 MB)
  [2/3] Sentinel-1 SAR Flood Mask...
       OK (0.01 MB)
  [3/3] MODIS NDVI (2015-01-01 to 2015-02-15)...
       OK (0.04 MB)

[2/9] Moderate Floods (Feb 2017) - MINOR
  [1/3] CHIRPS Rainfall...
       OK (0.00 MB)
  [2/3] Sentinel-1 SAR Flood Mask...
       OK (0.01 MB)
  [3/3] MODIS NDVI (2017-01-15 to 2017-03-01)...
       OK (0.04 MB)

[3/9] Cyclone Idai (Mar 2019) - MAJOR
  [1/3] CHIRPS Rainfall...
       OK (0.00 MB)
  [2/3] Sentinel-1 SAR Flood Mask...
       OK (0.01 MB)
  [3/3] MODIS NDVI (2019-02-15 to 2019-04-01)...
       OK (0.04 MB)

[4/9] Cyclone Ana (Jan 2022) - MAJOR
  [1/3] CHIRPS Rainfall...
       OK (0.00 MB)
  [2/3] Sentinel-1 SAR Flood Mask...
       OK (0.01 MB)
  [3/3] MODIS NDVI (2022-01-01 to 2022-02-15)...
       OK (0.04 MB)

[5/9] Cyclone Freddy (Mar 2023) - MAJOR
  [1/3] CHIRPS Rainfall...
       OK (0.00 MB)
  [2/3] Sentinel-1 SAR Flood Mask...
       OK (0.01 MB)
  [3